# TIM: where should the LoRA adapter go?

Everyone adapts `q_proj` and `v_proj`. That is a convention from the original LoRA paper, and it knows nothing about **your** two tasks.

This notebook measures, **before training anything**, how much the gradient subspaces of your two tasks overlap in each module, and hands you a list you can paste straight into `LoraConfig(target_modules=...)`.

It trains nothing. No optimiser, no parameter updates. A few forward and backward passes.

**Hardware.** The free Colab T4 (16 GB GPU, ~13 GB RAM) is enough for models up to about 1.5B. A 3B is borderline on RAM. **An 8B needs an A100 or a 40 GB card**, because the gradients of every target module are held in RAM. Do not quantise to save memory: you would be measuring the gradients of a different model.

**Paper:** https://doi.org/10.5281/zenodo.21999659 · **Code:** https://github.com/BiomeMakers/TIM-OmegaS

The measurement code is fetched from that repository, unmodified, so this notebook cannot drift from the published method.

In [ ]:
# DO NOT install or upgrade torch here. Colab ships torch and torchvision as a
# matched pair; upgrading one breaks the other and transformers then fails with
# "operator torchvision::nms does not exist".
# datasets is pinned because >=3 dropped script-based datasets, and two of the
# task sources here are script-based.
!pip -q install "datasets==2.21.0" "fsspec<=2024.6.1" -U transformers accelerate 2>&1 | tail -3

import torch, datasets, transformers, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('datasets', datasets.__version__, '| transformers', transformers.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'{d.name}, {d.total_memory/1e9:.0f} GB, compute capability {d.major}.{d.minor}')
    print('bf16 supported:', d.major >= 8)
else:
    print('NO GPU. Runtime > Change runtime type > GPU. On CPU this takes hours.')

# If pip changed anything above: Runtime > Restart session, then run from here.
from transformers import AutoConfig
AutoConfig  # if this import raised, restart the runtime before going on
print('\nimports OK')

## 1. Choose your model and your two tasks

`TASK_A` is what the model already does and you want it to keep. `TASK_B` is what you are about to fine-tune on.

In [ ]:
MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # @param {type:"string"}
TASK_A  = "code"   # @param ["code", "prose", "math", "legal"]
TASK_B  = "prose"  # @param ["code", "prose", "math", "legal"]
RANK    = 8        # @param {type:"integer"}
N_BATCH = 8        # @param {type:"integer"}
TARGETS = "q_proj,k_proj,v_proj,o_proj"  # @param {type:"string"}

# Other models that fit on a free T4:
#   meta-llama/Llama-3.2-1B          (gated, needs a token)
#   Qwen/Qwen2.5-1.5B
#   HuggingFaceTB/SmolLM2-1.7B
# 8B, needs A100 / 40 GB:
#   NousResearch/Meta-Llama-3-8B     (ungated mirror)
#
# RANK must be the LoRA rank you actually intend to train with: the measurement
# compares the rank-r subspaces an adapter of that size would use.
# N_BATCH=8 is enough to see the ranking. Raise it to 32 to check stability.
# Add MLP modules with e.g. TARGETS="q_proj,k_proj,v_proj,o_proj,down_proj".
assert TASK_A != TASK_B, 'The two tasks must be different.'
print(f'{MODEL}: {TASK_A} -> {TASK_B}, rank {RANK}, {N_BATCH} batches')

## 2. Fetch the measurement, unmodified

One patch is applied and it is printed below so you can see it: the published script asks for bfloat16 on any CUDA device, and the T4 does not support it. On cards older than Ampere it falls back to float16.

In [ ]:
import urllib.request, hashlib, re
URL = 'https://raw.githubusercontent.com/BiomeMakers/TIM-OmegaS/main/experiments/adapter_placement.py'
src = urllib.request.urlopen(URL).read().decode()
print('source :', URL)
print('sha256 :', hashlib.sha256(src.encode()).hexdigest())

old = '    DEV, DTYPE = "cuda", torch.bfloat16'
new = ('    DEV = "cuda"\n'
       '    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16')
assert old in src, 'The upstream script changed; check it before trusting this patch.'
src = src.replace(old, new, 1)
open('adapter_placement.py', 'w').write(src)
print('\npatch applied:\n', old, '\n->\n', new)

In [ ]:
# Preflight: 20 seconds, and it fails here with a clear message instead of
# after downloading the model.
from datasets import load_dataset
SOURCES = {'code':  ('code-search-net/code_search_net', 'python', 'whole_func_string'),
           'prose': ('Skylion007/openwebtext', None, 'text'),
           'math':  ('open-r1/OpenR1-Math-220k', None, 'problem'),
           'legal': ('pile-of-law/pile-of-law', 'r_legaladvice', 'text')}
for t in (TASK_A, TASK_B):
    repo, cfg, field = SOURCES[t]
    try:
        ds = (load_dataset(repo, cfg, split='train', streaming=True, trust_remote_code=True)
              if cfg else
              load_dataset(repo, split='train', streaming=True, trust_remote_code=True))
        ex = next(iter(ds))[field]
        print(f'OK  {t:6} {repo}  ({len(ex)} chars in the first example)')
    except Exception as e:
        print(f'FAIL {t:6} {repo}\n     {type(e).__name__}: {str(e)[:300]}')
        print('     Pick another task, or re-run cell 1 and restart the runtime.')

## 3. Run it

Minutes on a 1B model. The output ends with a ranking and a suggested placement, which is the quartile of largest relative drop among the modules that discriminate.

In [ ]:
import os, sys, subprocess
env = dict(os.environ, MODEL=MODEL, TASK_A=TASK_A, TASK_B=TASK_B,
           RANK=str(RANK), N_BATCH=str(N_BATCH), TARGETS=TARGETS,
           OUT='placement.json', PYTHONUNBUFFERED='1')

# Streamed line by line: a silent cell for ten minutes is indistinguishable
# from a hung one, and the first run downloads both the model and the data.
proc = subprocess.Popen([sys.executable, '-u', 'adapter_placement.py'], env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()

if rc != 0 or not os.path.exists('placement.json'):
    raise SystemExit(
        f'\nThe measurement did not finish (exit code {rc}). The error is in the '
        'output above.\nIf it is a dataset loading error, re-run cell 1 (the pinned '
        'versions) and\nrestart the runtime, or pick a different TASK_A / TASK_B pair.')
print('\nplacement.json written:', os.path.getsize('placement.json'), 'bytes')

## 4. The list, ready to paste

This is the point of the notebook. Everything above is how it was obtained.

In [ ]:
import json
rows = json.load(open('placement.json'))          # already sorted by rel, best first
useful = [r for r in rows if not r['saturated']]  # saturated ceilings cannot discriminate
k = max(1, len(useful) // 4)
sel = [r['module'].replace('.weight', '') for r in useful[:k]]

print(f'{len(rows)} modules measured, {len(rows)-len(useful)} saturated, '
      f'suggested placement = top quartile = {k} modules\n')
from collections import Counter
print('composition:', dict(Counter(m.split(".")[-1] for m in sel)))
depth = [int(m.split('.')[2]) for m in sel if m.split('.')[2].isdigit()]
if depth:
    print(f'layer depth: mean {sum(depth)/len(depth):.1f}, range {min(depth)}-{max(depth)}')

print('\n' + '=' * 70)
print('from peft import LoraConfig')
print(f'config = LoraConfig(')
print(f'    r={RANK}, lora_alpha={2*RANK}, lora_dropout=0.05, bias="none",')
print('    target_modules=[')
for m in sel:
    print(f'        "{m}",')
print('    ],\n)')
print('=' * 70)

# A fair comparison keeps the module COUNT equal between arms. The conventional
# arm must be built WITHOUT the diagnostic, or it is not a baseline: q_proj and
# v_proj on layers spread evenly through the model, cut to the same count.
import re
layers = sorted({int(m.split('.')[2]) for m in
                 (r['module'] for r in rows) if m.split('.')[2].isdigit()})
conv, i = [], 0
while len(conv) < k and i < len(layers) * 2:
    L = layers[round(i / 2 * (len(layers) - 1) / max(1, (k / 2) - 1)) % len(layers)] \
        if k > 2 else layers[i % len(layers)]
    m = f'model.layers.{L}.self_attn.' + ('q_proj' if i % 2 == 0 else 'v_proj')
    if m not in conv:
        conv.append(m)
    i += 1
print(f'\nconventional arm, same count, built without the diagnostic: {len(conv)} modules')
print('  layers:', sorted({int(m.split(".")[2]) for m in conv}))

json.dump({'suggested': sel, 'conventional_same_count': conv,
           'model': MODEL, 'task_a': TASK_A, 'task_b': TASK_B,
           'rank': RANK, 'n_batch': N_BATCH},
          open('tim_placement_ready.json', 'w'), indent=2)
try:
    from google.colab import files
    files.download('tim_placement_ready.json')
except Exception:
    pass

## How to read it, and what it does not tell you

**A high ceiling is not a bad module.** It means that module's gradients are dictated by the structure of the model rather than by the task, so it cannot discriminate. Those are excluded from the ranking, not condemned.

**Check stability before you trust it.** Re-run with `N_BATCH=32`. If the ceiling rises, the first run was sampling noise. If the ranking holds, it is structure.

**What is measured and what is not.** On Llama-3-8B, code to prose, the measured placement retained 75.6% against 54.1% for the default over ten paired seeds, nine of ten. That is one model and one task pair. The type ordering (`o_proj` first, `v_proj` last) reproduced on Mistral-7B and Qwen2.5-7B; the concentration in deep layers did not, so depth is a property of the base model. And the honest rival is not the `q,v` default but a cheap rule over module type and depth: it ranks well, it selects a different set in about a third of the modules, and whether that changes retention is a training run in progress.

**If you run this, please post what you get, including nothing.** A null on your model and your task pair is more useful to us than another positive on ours: https://github.com/BiomeMakers/TIM-OmegaS/issues

Measuring is free under the repository licence. Paper: https://doi.org/10.5281/zenodo.21999659